# 补充分析：每类指标、三张图、计算开销、越南外部数据集

接着昨晚的 160 次训练做。**前 6 格和通宵笔记本完全一样**（读取缓存和划分，定义模型和训练函数），不会重新训练那 160 次。

| 部分 | 内容 | 大概时间 |
|---|---|---|
| A | 每类 precision / recall / F1 + 混淆矩阵 | 几秒 |
| B | 三张图：每折配对差、只换种子的差距、折与折的差别 | 几秒 |
| C | 参数量、计算量、CPU 推理时间 | 几分钟 |
| D | 越南外部数据集：用全部 550 张训练 4 配置 × 5 种子，直接测越南 | 约 40 分钟 |

结果都存在 Drive 的 `runs_phase0/analysis/`。D 部分可以断了续跑。
运行时选 GPU（C 部分只用 CPU，不受影响）。全部跑完后，把 A、C、D3 的输出和三张图发给 Claude。

## 1. 挂载 Drive，检查 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout or '⚠ 没有 GPU！请改运行时类型')

## 2. 设置

In [ ]:
from pathlib import Path

DRIVE_DIR    = Path('/content/drive/MyDrive/p1_reconstructure')   # 你上传 data_original.zip 的文件夹
DATA_ARCHIVE = DRIVE_DIR / 'data_original.zip'
OUT_DIR      = DRIVE_DIR / 'runs_phase0'          # 结果写在这里（每跑完一次就保存）
LOCAL_DIR    = Path('/content/work')               # Colab 本地临时目录

DROP_PINK    = True          # 去掉 Pink disease（只有 3 个 session）
CROSS_CLASS  = True          # 按跨类别规则合并 session（同一段连拍被分到两个类别时合成一组）
N_FOLDS      = 4
N_REPEATS    = 5             # 4 折 × 5 次 = 20 折
SEEDS        = [0, 1]        # 每折每个配置的训练种子
PHASES       = [['none', 'lfa'], ['se', 'cbam']]   # 先跑完第一组，再跑第二组
EPOCHS_S1, EPOCHS_S2 = 15, 15
BATCH        = 16
INNER_VAL_FRAC = 0.1         # 训练折内再按组分出 10% 做验证集（选 checkpoint 用）
PRETRAINED   = True          # 本地测试时可设 False
QUICK_TEST   = False         # True = 每阶段 1 个 epoch、只跑 2 折，用来先试跑几分钟

## 3. 解压原图，缩放到 512 px，缓存到 Drive（断线重连后不用再缩放）

In [ ]:
import io, os, re, shutil, tarfile, zipfile, pickle, time
from PIL import Image

CLASSES_ALL = ['Algal', 'Leaf_rot', 'Phomopsis', 'Pink_disease', 'Root_disease']
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff', '.heic'}
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE = OUT_DIR / 'cache_512.pkl'


def norm_class(part):
    if part.startswith('Leaf_rot'):
        return 'Leaf_rot'
    for c in CLASSES_ALL:
        if part.lower() == c.lower():
            return c
    return None


def shrink_like_paper(raw_bytes):
    """和 shrink.py 完全一致：长边 512，Lanczos，JPEG q92。"""
    with Image.open(io.BytesIO(raw_bytes)) as im:
        im.load()
        if im.mode in ('RGBA', 'P', 'LA'):
            im = im.convert('RGB')
        w, h = im.size
        s = 512 / max(w, h)
        if s < 1:
            im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)
        buf = io.BytesIO()
        im.save(buf, 'JPEG', quality=92, optimize=True)
        return buf.getvalue()


if CACHE.exists():
    with open(CACHE, 'rb') as f:
        cache = pickle.load(f)
    print(f'读取缓存：{len(cache["images"])} 张 512 px 图片')
else:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print('从 Drive 复制压缩包到本地……')
    local_arc = LOCAL_DIR / 'data.arc'
    shutil.copy(DATA_ARCHIVE, local_arc)
    out = LOCAL_DIR / 'data_original'
    if zipfile.is_zipfile(local_arc):
        zipfile.ZipFile(local_arc).extractall(out)
    elif tarfile.is_tarfile(local_arc):
        tarfile.open(local_arc).extractall(out)
    else:
        raise SystemExit('压缩包既不是 zip 也不是 tar')
    sess_files = sorted(out.rglob('sessions.csv'), key=lambda p: len(p.parts))
    assert sess_files, '压缩包里没找到 sessions.csv'
    images = {}
    files = [p for p in out.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT]
    for i, p in enumerate(files, 1):
        cls = next((norm_class(x) for x in p.relative_to(out).parts[:-1] if norm_class(x)), None)
        if cls is None:
            continue
        images[(cls, p.name)] = shrink_like_paper(p.read_bytes())
        if i % 50 == 0:
            print(f'  缩放 {i}/{len(files)}', end='\r')
    cache = {'images': images, 'sessions_csv': sess_files[0].read_text(encoding='utf-8-sig')}
    with open(CACHE, 'wb') as f:
        pickle.dump(cache, f)
    print(f'\n缩放完成：{len(images)} 张，用时 {time.time() - t0:.0f} 秒，已缓存到 Drive')

assert len(cache['images']) == 560, f'应该是 560 张，实际 {len(cache["images"])}'

## 4. session 分组和 20 折划分（第一次生成后存成 folds.csv，之后一直复用）

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

RE_SEQ = re.compile(r'^(?P<prefix>[A-Za-z_]*?)(?P<num>\d{3,6})\b')


def build_groups(d, gap=3, cross_class=True):
    """和 grouping_sensitivity.py 的 build() 相同：连拍按编号间隔 ≤ gap 合并。"""
    d = d.copy()
    d['kind'] = d['session'].astype(str).str.split(':').str[0]
    parsed = d['file'].apply(lambda f: RE_SEQ.match(str(f)))
    d['prefix'] = [(m.group('prefix') or 'IMG') if m else None for m in parsed]
    d['num'] = [int(m.group('num')) if m else np.nan for m in parsed]
    seq = d['kind'].isin(['burst', 'single']) & d['num'].notna()
    lab = {}
    key = ['prefix'] if cross_class else ['cls', 'prefix']
    for k, grp in d[seq].groupby(key):
        run, prev = 0, None
        for i, r in grp.sort_values('num').iterrows():
            if prev is not None and r['num'] - prev > gap:
                run += 1
            lab[i] = f'seq:{k}:{run}'
            prev = r['num']
    for i, r in d[~seq].iterrows():
        lab[i] = r['session']
    return pd.Series(lab).reindex(d.index)


meta = pd.read_csv(io.StringIO(cache['sessions_csv']))
meta['cls'] = meta['cls'].map(norm_class)
meta['group'] = build_groups(meta, cross_class=CROSS_CLASS) if CROSS_CLASS else meta['session']
if DROP_PINK:
    meta = meta[meta['cls'] != 'Pink_disease'].reset_index(drop=True)
CLASSES = sorted(meta['cls'].unique(), key=CLASSES_ALL.index)
meta['y'] = meta['cls'].map({c: i for i, c in enumerate(CLASSES)})
missing = [(c, f) for c, f in zip(meta['cls'], meta['file']) if (c, f) not in cache['images']]
assert not missing, f'sessions.csv 里有 {len(missing)} 个文件在图片里找不到，例如 {missing[:3]}'
print(f'{len(meta)} 张图，{meta["group"].nunique()} 组，类别 {CLASSES}')
print(meta.groupby('cls')['group'].nunique().rename('组数').to_string())

FOLDS_CSV = OUT_DIR / 'folds.csv'
if FOLDS_CSV.exists():
    folds = pd.read_csv(FOLDS_CSV)
    print(f'复用已有划分：{FOLDS_CSV.name}')
else:
    rows = []
    for rep in range(N_REPEATS):
        sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=rep)
        for k, (tr, te) in enumerate(sgkf.split(meta, meta['y'], meta['group'])):
            # 训练折内再按组分出验证集
            # 组的大小很不均匀（最大一组上百张），所以在 10 种候选里挑最接近 10% 且至少含 3 类的那种
            inner = StratifiedGroupKFold(n_splits=round(1 / INNER_VAL_FRAC), shuffle=True,
                                         random_state=1000 + rep * 10 + k)
            sub = meta.iloc[tr]
            target = INNER_VAL_FRAC * len(sub)
            cands = [(abs(len(v) - target) + (1e6 if sub.iloc[v]['y'].nunique() < 3 else 0), t, v)
                     for t, v in inner.split(sub, sub['y'], sub['group'])]
            _, itr, iva = min(cands, key=lambda c: c[0])
            role = np.array(['test'] * len(meta), dtype=object)
            role[tr[itr]] = 'train'
            role[tr[iva]] = 'val'
            for idx in range(len(meta)):
                rows.append((rep, k, idx, role[idx]))
    folds = pd.DataFrame(rows, columns=['repeat', 'fold', 'idx', 'role'])
    folds.to_csv(FOLDS_CSV, index=False)
    print(f'已生成并保存划分：{FOLDS_CSV.name}')

# 检查：没有任何组同时出现在训练和测试里
for (rep, k), f in folds.groupby(['repeat', 'fold']):
    g = {r: set(meta.loc[f[f.role == r].idx, 'group']) for r in ['train', 'val', 'test']}
    assert not (g['train'] & g['test']) and not (g['val'] & g['test']) and not (g['train'] & g['val']), (rep, k)
cnt = folds.groupby(['repeat', 'fold', 'role']).size().unstack()
print(f'每折图片数（平均）：train {cnt.train.mean():.0f} / val {cnt.val.mean():.0f} / test {cnt.test.mean():.0f}，'
      '所有折都没有组跨越 train/val/test ✓')

## 5. 模型（与 train.py 完全相同）

In [ ]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


class LesionFocusAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attention_conv = nn.Conv2d(in_channels, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return x * self.sigmoid(self.attention_conv(x))


class SEModule(nn.Module):
    def __init__(self, in_ch, reduction=16):
        super().__init__()
        mid = max(in_ch // reduction, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(in_ch, mid, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(mid, in_ch, bias=False), nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)


class CBAMModule(nn.Module):
    def __init__(self, in_ch, reduction=16, ks=7):
        super().__init__()
        mid = max(in_ch // reduction, 1)
        self.avg, self.mx = nn.AdaptiveAvgPool2d(1), nn.AdaptiveMaxPool2d(1)
        self.cfc = nn.Sequential(nn.Linear(in_ch, mid, bias=False), nn.ReLU(inplace=True),
                                 nn.Linear(mid, in_ch, bias=False))
        self.sconv = nn.Conv2d(2, 1, kernel_size=ks, padding=ks // 2, bias=False)
        self.sig = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        ch = self.sig(self.cfc(self.avg(x).view(b, c)) + self.cfc(self.mx(x).view(b, c))).view(b, c, 1, 1)
        x = x * ch
        sp = self.sig(self.sconv(torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1)))
        return x * sp


def build_model(attention, n_cls):
    m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if PRETRAINED else None)
    for p in m.parameters():
        p.requires_grad = False
    last = m.features[-1][0].out_channels
    att = {'lfa': LesionFocusAttention, 'se': SEModule, 'cbam': CBAMModule}.get(attention)
    if att:
        m.features = nn.Sequential(m.features, att(last))
    m.classifier = nn.Sequential(nn.Dropout(p=0.3, inplace=True), nn.Linear(m.classifier[1].in_features, n_cls))
    for p in m.classifier.parameters():
        p.requires_grad = True
    if att:
        for p in m.features[-1].parameters():
            p.requires_grad = True
    return m


for a in ['none', 'lfa', 'se', 'cbam']:
    extra = sum(p.numel() for p in build_model(a, 4).parameters()) - sum(p.numel() for p in build_model('none', 4).parameters())
    print(f'{a:<5} 额外参数 {extra:,}')

## 6. 训练函数（种子真正生效，按 macro F1 选 checkpoint）

In [ ]:
import copy
from sklearn.metrics import f1_score, accuracy_score

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomAffine(degrees=20, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.GaussianBlur(kernel_size=(3, 7), sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.10)),
])
test_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

PIL_CACHE = {k: Image.open(io.BytesIO(v)).convert('RGB') for k, v in cache['images'].items()}


class DS(Dataset):
    def __init__(self, idx, tf):
        self.idx, self.tf = list(idx), tf

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        r = meta.iloc[self.idx[i]]
        return self.tf(PIL_CACHE[(r['cls'], r['file'])]), int(r['y'])


def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _worker_init(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    yt, yp = [], []
    for X, y in loader:
        with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
            out = model(X.to(DEVICE))
        yp += out.argmax(1).cpu().tolist()
        yt += y.tolist()
    return np.array(yt), np.array(yp)


def macro_f1_present(yt, yp):
    """只对验证集里出现的类别算 macro F1（小验证集可能缺某一类）。"""
    return f1_score(yt, yp, labels=sorted(set(yt)), average='macro', zero_division=0)


def run_one(attention, tr_idx, va_idx, te_idx, seed):
    seed_everything(seed)
    n_cls = len(CLASSES)
    ytr = meta.iloc[tr_idx]['y'].values
    counts = np.bincount(ytr, minlength=n_cls)
    cw = torch.tensor(len(ytr) / (n_cls * np.maximum(counts, 1)), dtype=torch.float, device=DEVICE)
    sw = (1.0 / np.maximum(counts, 1))[ytr]
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True, generator=g)
    kw = dict(num_workers=2, worker_init_fn=_worker_init, pin_memory=(DEVICE == 'cuda'))
    tl = DataLoader(DS(tr_idx, train_tf), batch_size=BATCH, sampler=sampler, generator=g, **kw)
    vl = DataLoader(DS(va_idx, test_tf), batch_size=64, shuffle=False, **kw)
    te = DataLoader(DS(te_idx, test_tf), batch_size=64, shuffle=False, **kw)

    model = build_model(attention, n_cls).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    best = (-1.0, None)
    e1, e2 = (1, 1) if QUICK_TEST else (EPOCHS_S1, EPOCHS_S2)

    def epochs(n, lr):
        nonlocal best
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
        for _ in range(n):
            model.train()
            for X, y in tl:
                X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
                    loss = crit(model(X), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            f1v = macro_f1_present(*predict(model, vl))
            if f1v > best[0]:
                best = (f1v, copy.deepcopy(model.state_dict()))

    epochs(e1, 1e-3)                       # 阶段 1：冻结骨干，只训练注意力和分类头
    model.load_state_dict(best[1])
    for p in model.parameters():
        p.requires_grad = True
    epochs(e2, 1e-5)                       # 阶段 2：全部解冻
    model.load_state_dict(best[1])
    yt, yp = predict(model, te)
    return {
        'val_macro_f1': round(best[0] * 100, 2),
        'test_macro_f1': round(f1_score(yt, yp, labels=list(range(n_cls)), average='macro', zero_division=0) * 100, 2),
        'test_acc': round(accuracy_score(yt, yp) * 100, 2),
    }, yt, yp

## A. 每类指标（不用训练，几秒钟）
用 160 次训练保存下来的 preds.csv，算每个配置在每一类上的 precision / recall / F1（40 次运行的均值 ± SD），
再把 40 次运行的混淆矩阵加起来，按行归一化。回答 IPA 审稿人"要每类指标"那条。

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

RES_DIR = OUT_DIR / 'main'
ANA_DIR = OUT_DIR / 'analysis'
ANA_DIR.mkdir(parents=True, exist_ok=True)
CONFIGS = ['none', 'lfa', 'se', 'cbam']
LABEL = {'none': 'B0 (no attention)', 'lfa': 'B0 + LFA', 'se': 'B0 + SE', 'cbam': 'B0 + CBAM'}

# 图的统一样式（数据可视化规范：细线、浅网格、文字用中性色）
INK, INK2, GRID, SURF = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
BLUE, ORANGE = '#2a78d6', '#eb6834'
plt.rcParams.update({'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF,
                     'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'text.color': INK,
                     'xtick.color': INK2, 'ytick.color': INK2, 'axes.grid': True, 'grid.color': GRID,
                     'grid.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold'})

preds = pd.read_csv(RES_DIR / 'preds.csv')
res = pd.read_csv(RES_DIR / 'results.csv')
labels_idx = list(range(len(CLASSES)))
rows = []
for (cfg, rep, k, s), g in preds.groupby(['config', 'repeat', 'fold', 'seed']):
    p, r, f, n = precision_recall_fscore_support(g.y_true, g.y_pred, labels=labels_idx, zero_division=0)
    for i, c in enumerate(CLASSES):
        rows.append({'config': cfg, 'class': c, 'precision': p[i] * 100, 'recall': r[i] * 100,
                     'f1': f[i] * 100, 'support': n[i]})
pc = pd.DataFrame(rows)
summary = pc.groupby(['config', 'class'])[['precision', 'recall', 'f1']].agg(['mean', 'std']).round(1)
summary = summary.reindex(pd.MultiIndex.from_product([CONFIGS, CLASSES]))
summary.to_csv(ANA_DIR / 'per_class_metrics.csv')
print('每类指标（%，40 次运行的均值 / SD）')
print(summary.to_string())

# 合并混淆矩阵：无注意力 vs LFA
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
for ax, cfg in zip(axes, ['none', 'lfa']):
    g = preds[preds.config == cfg]
    cm = confusion_matrix(g.y_true, g.y_pred, labels=labels_idx)
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    ax.grid(False)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, f'{cmn[i, j] * 100:.0f}', ha='center', va='center',
                    color='white' if cmn[i, j] > 0.55 else INK, fontsize=10)
    ax.set_xticks(labels_idx, [c.replace('_', ' ') for c in CLASSES], rotation=30, ha='right')
    ax.set_yticks(labels_idx, [c.replace('_', ' ') for c in CLASSES])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{LABEL[cfg]} (row %, 40 runs pooled)')
fig.tight_layout()
fig.savefig(ANA_DIR / 'fig_confusion_none_vs_lfa.png', dpi=300)
plt.show()

## B. 三张图（不用训练）
1. 每折配对差：每个点是一折（20 折），横轴是"加模块 − 不加"的 macro F1 差，竖线是均值和校正 95% CI。
2. 只换种子差多少：同一配置、同一折，种子 0 和种子 1 的差的绝对值。
3. 折与折的差别：每条灰线是一折，四种配置的分数连起来——折之间的差距远大于配置之间。

In [ ]:
from scipy import stats

per_fold = res.groupby(['config', 'repeat', 'fold'])['test_macro_f1'].mean().unstack('config')[CONFIGS]
n_tr = (res['n_train'] + res['n_val']).mean()
n_te = res['n_test'].mean()


def corrected_ci(d):
    J = len(d)
    se = np.sqrt((1 / J + n_te / n_tr) * d.var(ddof=1))
    h = stats.t.ppf(0.975, J - 1) * se
    return d.mean(), d.mean() - h, d.mean() + h


# 图 1：每折配对差
fig, ax = plt.subplots(figsize=(8, 3.2))
comps = ['lfa', 'se', 'cbam']
rng = np.random.default_rng(0)
for yi, c in enumerate(comps):
    d = (per_fold[c] - per_fold['none']).values
    m, lo, hi = corrected_ci(d)
    ax.scatter(d, yi + rng.uniform(-0.12, 0.12, len(d)), s=22, color='#9a9a94', zorder=2,
               edgecolor=SURF, linewidth=0.8)
    ax.plot([lo, hi], [yi, yi], color=BLUE, lw=2.5, zorder=3, solid_capstyle='round')
    ax.scatter([m], [yi], s=70, color=BLUE, zorder=4, edgecolor=SURF, linewidth=2)
    ax.text(1.02, yi, f'{m:+.2f} [{lo:.1f}, {hi:.1f}]', va='center', color=INK2, fontsize=9,
            transform=ax.get_yaxis_transform())
ax.axvline(0, color=INK2, lw=1, zorder=1)
ax.set_yticks(range(len(comps)), [f'{LABEL[c]} − no attention' for c in comps])
ax.set_xlabel('Paired difference in macro F1 (pp), per fold')
ax.set_title('Attention module minus plain backbone, 20 grouped folds', loc='left')
ax.text(1.02, -0.55, 'mean [95% CI]', color=INK2, fontsize=9, fontweight='bold',
        transform=ax.get_yaxis_transform())
ax.grid(axis='y', visible=False)
xmax = max(abs((per_fold[c] - per_fold['none'])).max() for c in comps) + 1.5
ax.set_xlim(-xmax, xmax)
ax.set_ylim(len(comps) - 0.5, -0.75)
fig.tight_layout()
fig.savefig(ANA_DIR / 'fig1_paired_differences.png', dpi=300)
plt.show()

# 图 2：只换种子差多少
seed_gap = res.pivot_table(index=['config', 'repeat', 'fold'], columns='seed', values='test_macro_f1')
seed_gap = (seed_gap[SEEDS[0]] - seed_gap[SEEDS[1]]).abs().rename('gap').reset_index()
lfa_effect = abs((per_fold['lfa'] - per_fold['none']).mean())
fig, ax = plt.subplots(figsize=(8, 3.4))
for yi, c in enumerate(CONFIGS):
    v = seed_gap[seed_gap.config == c].gap.values
    ax.scatter(v, yi + rng.uniform(-0.12, 0.12, len(v)), s=22, color='#9a9a94',
               edgecolor=SURF, linewidth=0.8, zorder=2)
    ax.plot([np.median(v)] * 2, [yi - 0.25, yi + 0.25], color=BLUE, lw=2.5, zorder=3)
    ax.text(1.02, yi, f'median {np.median(v):.1f}', va='center', color=INK2, fontsize=9,
            transform=ax.get_yaxis_transform())
ax.axvline(lfa_effect, color=ORANGE, lw=1.5, ls='--', zorder=1)
ax.text(lfa_effect + 0.3, len(CONFIGS) - 0.45, f'|mean LFA effect| = {lfa_effect:.2f} pp',
        color=ORANGE, fontsize=9, va='center')
ax.set_yticks(range(len(CONFIGS)), [LABEL[c] for c in CONFIGS])
ax.set_xlabel('|seed 0 − seed 1| macro F1 (pp), same fold, same configuration')
ax.set_title('Score change from the random seed alone', loc='left')
ax.grid(axis='y', visible=False)
ax.set_xlim(left=0)
ax.set_ylim(-0.5, len(CONFIGS) - 0.2)
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(ANA_DIR / 'fig2_seed_variability.png', dpi=300)
plt.show()

# 图 3：折与折的差别
fig, ax = plt.subplots(figsize=(6, 3.8))
x = np.arange(len(CONFIGS))
for _, row in per_fold.iterrows():
    ax.plot(x, row.values, color='#bdbcb6', lw=1, zorder=1)
ax.plot(x, per_fold.mean().values, color=BLUE, lw=2.5, marker='o', ms=8,
        markeredgecolor=SURF, markeredgewidth=2, zorder=3, label='mean of 20 folds')
ax.set_xticks(x, [LABEL[c] for c in CONFIGS], rotation=15)
ax.set_ylabel('Macro F1 (%), mean of 2 seeds')
ax.set_title('Each grey line is one fold', loc='left')
ax.legend(frameon=False, loc='lower right')
ax.grid(axis='x', visible=False)
fig.tight_layout()
fig.savefig(ANA_DIR / 'fig3_fold_vs_config.png', dpi=300)
plt.show()
print(f'折间差距：{per_fold.mean(1).min():.1f} – {per_fold.mean(1).max():.1f}；'
      f'配置间差距：{per_fold.mean().min():.2f} – {per_fold.mean().max():.2f}')
print(f'图已存到 {ANA_DIR}')

## C. 参数量、计算量和 CPU 推理时间（几分钟）
batch = 1，224×224，四个模型轮流计时（每个共 200 次），取中位数。分单线程和全部线程两种。
回答 IPA 审稿人"LFA 的计算开销没有量化"那条。Colab 的 CPU 和你的电脑不同，绝对数值只作相对比较。

In [ ]:
import time, subprocess
from torch.utils.flop_counter import FlopCounterMode

lat_rows = []
x1 = torch.randn(1, 3, 224, 224)
base_params = sum(p.numel() for p in build_model('none', len(CLASSES)).parameters())
models_cpu = {c: build_model(c, len(CLASSES)).eval() for c in CONFIGS}
for cfg, m in models_cpu.items():
    params = sum(p.numel() for p in m.parameters())
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        m(x1)
    lat_rows.append({'config': cfg, 'params_M': round(params / 1e6, 4), 'extra_params': params - base_params,
                     'GMACs': round(fc.get_total_flops() / 2 / 1e9, 4)})   # 乘加算一次，按 GMACs 惯例
# 四个模型轮流计时（5 轮 × 每轮 40 次），避免机器状态随时间漂移只影响某一个模型
for threads in [1, os.cpu_count()]:
    torch.set_num_threads(threads)
    times = {c: [] for c in CONFIGS}
    with torch.no_grad():
        for m in models_cpu.values():
            for _ in range(30):
                m(x1)
        for _ in range(5):
            for c, m in models_cpu.items():
                for _ in range(40):
                    t0 = time.perf_counter(); m(x1); times[c].append((time.perf_counter() - t0) * 1000)
    for row in lat_rows:
        row[f'cpu_ms_{threads}thr_median'] = round(float(np.median(times[row['config']])), 2)
        row[f'cpu_ms_{threads}thr_p90'] = round(float(np.percentile(times[row['config']], 90)), 2)
torch.set_num_threads(os.cpu_count())
lat = pd.DataFrame(lat_rows)
lat.to_csv(ANA_DIR / 'latency_params.csv', index=False)
print('\n' + lat.to_string(index=False))
cpu_name = subprocess.run(['bash', '-c', "lscpu | grep 'Model name'"], capture_output=True, text=True).stdout.strip()
print(f'\n（CPU：{cpu_name}，{os.cpu_count()} 线程）')

## D. 外部数据集：越南榴莲叶片数据（约 40 分钟）
用全部 550 张马来西亚图训练（每个配置 5 个种子，共 20 次），直接拿去测越南数据，不做任何微调。
只用三个能对应上的叶片类别：Leaf_Algal→Algal，Leaf_Blight / Leaf_Colletotrichum→Leaf_rot，Leaf_Phomopsis→Phomopsis。
（Leaf_Rhizoctonia→Root_disease 在旧版里已验证对不上，不用。）

报告两种算法：
- **full**：模型照常在 4 类里选，选成 Root_disease 就算错；
- **3-class**：只在三个对应类别里选（把 Root_disease 屏蔽掉）。

注意：越南数据是去背景、统一缩放过的，和田间原图差别很大，分数会明显低于马来西亚内部的结果。
这里关心的是"换了数据来源以后，LFA 有没有帮助"，不是绝对分数。

In [ ]:
import hashlib
from collections import Counter

VN_ARCHIVE = DRIVE_DIR / 'Vietnam dataset.zip'     # 文件名不同就改这里
VN_MAP = {'leaf_algal': 'Algal', 'leaf_blight': 'Leaf_rot', 'leaf_colletotrichum': 'Leaf_rot',
          'leaf_phomopsis': 'Phomopsis'}
VN_CACHE = OUT_DIR / 'cache_vietnam_512.pkl'
EXT_SEEDS = [0, 1, 2, 3, 4]

if VN_CACHE.exists():
    with open(VN_CACHE, 'rb') as f:
        vn = pickle.load(f)
    print(f'读取越南缓存：{len(vn)} 张')
else:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    vdir = LOCAL_DIR / 'vietnam'
    vdir.mkdir(parents=True, exist_ok=True)
    shutil.copy(VN_ARCHIVE, LOCAL_DIR / 'vn.arc')
    arc = LOCAL_DIR / 'vn.arc'
    (zipfile.ZipFile(arc).extractall(vdir) if zipfile.is_zipfile(arc) else tarfile.open(arc).extractall(vdir))
    vn, seen, skipped = [], set(), Counter()
    for p in sorted(vdir.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in IMG_EXT:
            continue
        folder = next((x for x in reversed(p.relative_to(vdir).parts[:-1]) if x.lower() in VN_MAP), None)
        if folder is None:
            skipped[p.parent.name] += 1
            continue
        raw = p.read_bytes()
        h = hashlib.md5(raw).hexdigest()
        if h in seen:                       # 去掉完全重复的文件（比如 train/test 里各放了一份）
            skipped['(duplicate)'] += 1
            continue
        seen.add(h)
        vn.append({'vn_class': folder, 'cls': VN_MAP[folder.lower()], 'file': str(p.relative_to(vdir)),
                   'jpg': shrink_like_paper(raw)})
    with open(VN_CACHE, 'wb') as f:
        pickle.dump(vn, f)
    print(f'越南数据：保留 {len(vn)} 张；没用上的文件夹：{dict(skipped)}')

vn_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'jpg'} for r in vn])
print(vn_df.groupby(['cls', 'vn_class']).size().rename('张数').to_string())
VN_PIL = [Image.open(io.BytesIO(r['jpg'])).convert('RGB') for r in vn]
VN_Y = np.array([CLASSES.index(r['cls']) for r in vn])


class VNDS(Dataset):
    def __len__(self):
        return len(VN_PIL)

    def __getitem__(self, i):
        return test_tf(VN_PIL[i]), int(VN_Y[i])

## D2. 用全部马来西亚数据训练，测越南

In [ ]:
@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    out = []
    for X, _ in loader:
        with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
            out.append(model(X.to(DEVICE)).float().cpu())
    return torch.cat(out).numpy()


def train_model(attention, tr_idx, va_idx, seed):
    """和 run_one 相同的训练流程，只是返回训练好的模型。"""
    seed_everything(seed)
    n_cls = len(CLASSES)
    ytr = meta.iloc[tr_idx]['y'].values
    counts = np.bincount(ytr, minlength=n_cls)
    cw = torch.tensor(len(ytr) / (n_cls * np.maximum(counts, 1)), dtype=torch.float, device=DEVICE)
    sw = (1.0 / np.maximum(counts, 1))[ytr]
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True, generator=g)
    kw = dict(num_workers=2, worker_init_fn=_worker_init, pin_memory=(DEVICE == 'cuda'))
    tl = DataLoader(DS(tr_idx, train_tf), batch_size=BATCH, sampler=sampler, generator=g, **kw)
    vl = DataLoader(DS(va_idx, test_tf), batch_size=64, shuffle=False, **kw)
    model = build_model(attention, n_cls).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    best = [-1.0, None]
    e1, e2 = (1, 1) if QUICK_TEST else (EPOCHS_S1, EPOCHS_S2)
    for n_ep, lr, unfreeze in [(e1, 1e-3, False), (e2, 1e-5, True)]:
        if unfreeze:
            model.load_state_dict(best[1])
            for p in model.parameters():
                p.requires_grad = True
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
        for _ in range(n_ep):
            model.train()
            for X, y in tl:
                X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
                    loss = crit(model(X), y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            f1v = macro_f1_present(*predict(model, vl))
            if f1v > best[0]:
                best = [f1v, copy.deepcopy(model.state_dict())]
    model.load_state_dict(best[1])
    return model, best[0]


# 全部 550 张：按组分出约 10% 做验证（选 checkpoint 用），其余训练
inner = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=2026)
cands = [(abs(len(v) - 0.1 * len(meta)) + (1e6 if meta.iloc[v]['y'].nunique() < 3 else 0), t, v)
         for t, v in inner.split(meta, meta['y'], meta['group'])]
_, ext_tr, ext_va = min(cands, key=lambda c: c[0])
print(f'外部实验训练集 {len(ext_tr)} 张 / 验证集 {len(ext_va)} 张')

EXT_CSV = ANA_DIR / 'external_vietnam.csv'
done_ext = set()
if EXT_CSV.exists():
    e0 = pd.read_csv(EXT_CSV)
    done_ext = set(zip(e0.config, e0.seed))
vl_vn = DataLoader(VNDS(), batch_size=64, shuffle=False, num_workers=2)
mapped = [CLASSES.index(c) for c in ['Algal', 'Leaf_rot', 'Phomopsis']]
CKPT_DIR = ANA_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

for s in EXT_SEEDS:
    for cfg in CONFIGS:
        if (cfg, s) in done_ext:
            continue
        t0 = time.time()
        model, vf1 = train_model(cfg, ext_tr, ext_va, s)
        logits = predict_logits(model, vl_vn)
        yp_full = logits.argmax(1)
        masked = logits.copy()
        masked[:, [i for i in range(len(CLASSES)) if i not in mapped]] = -1e9
        yp_3 = masked.argmax(1)
        row = {'config': cfg, 'seed': s, 'val_macro_f1': round(vf1 * 100, 2), 'n_vietnam': len(VN_Y),
               'vn_macro_f1_full': round(f1_score(VN_Y, yp_full, labels=mapped, average='macro', zero_division=0) * 100, 2),
               'vn_macro_f1_3class': round(f1_score(VN_Y, yp_3, labels=mapped, average='macro', zero_division=0) * 100, 2),
               'vn_pred_as_root_pct': round((yp_full == CLASSES.index('Root_disease')).mean() * 100, 1),
               'minutes': round((time.time() - t0) / 60, 2)}
        pd.DataFrame([row]).to_csv(EXT_CSV, mode='a', header=not EXT_CSV.exists(), index=False)
        if s == EXT_SEEDS[0]:                    # 每个配置存一个模型，以后做 Grad-CAM 用
            torch.save(model.state_dict(), CKPT_DIR / f'{cfg}_seed{s}_all550.pth')
        print(f'{cfg:<5} seed{s}  越南 F1 full {row["vn_macro_f1_full"]:5.1f} / 3-class {row["vn_macro_f1_3class"]:5.1f}'
              f'  ({row["minutes"]:.1f} 分钟)', flush=True)

## D3. 外部数据集结果

In [ ]:
ext = pd.read_csv(EXT_CSV)
print(ext.groupby('config')[['vn_macro_f1_full', 'vn_macro_f1_3class', 'vn_pred_as_root_pct']]
      .agg(['mean', 'std']).round(2).reindex(CONFIGS).to_string())
print('\n配对比较（同一种子配对；这里的 p 只反映训练随机性，越南测试集是固定的一份）：')
for col in ['vn_macro_f1_full', 'vn_macro_f1_3class']:
    piv = ext.pivot_table(index='seed', columns='config', values=col)
    for c in ['lfa', 'se', 'cbam']:
        if c in piv and 'none' in piv:
            d = (piv[c] - piv['none']).dropna()
            if len(d) >= 3:
                print(f'  {col:<20} {c:<5}− none：平均差 {d.mean():+.2f}，配对 t 检验 p = {stats.ttest_rel(piv[c][d.index], piv["none"][d.index]).pvalue:.3f}，'
                      f'赢 {(d > 0).sum()}/{len(d)}')
print(f'\n三类均匀乱猜的基线约 33.3%。结果存在 {EXT_CSV}')